In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
X_train = pd.read_csv('../Datos/preparados/TrainX.csv')
y_train = pd.read_csv('../Datos/preparados/TrainY.csv')
X_val = pd.read_csv('../Datos/preparados/ValidationX.csv') 
y_val = pd.read_csv('../Datos/preparados/ValidationY.csv')
X_test = pd.read_csv('../Datos/preparados/TestX.csv')
y_test = pd.read_csv('../Datos/preparados/TestY.csv')

In [3]:
def crear_categorias_rentabilidad(y):
    return (y >= 1).astype(int)

In [ ]:
y_train_class = crear_categorias_rentabilidad(y_train)
y_val_class = crear_categorias_rentabilidad(y_val)
y_test_class = crear_categorias_rentabilidad(y_test)

print("DISTRIBUCIÓN DE CLASES:")
print(f" Rentabilidad baja: {(y_train_class == 0).sum()}")
print(f" Rentabilidad alta: {(y_train_class == 1).sum()}")


📊 DISTRIBUCIÓN DE CLASES:
   • Train - Rentabilidad baja: ROI    186508
dtype: int64
   • Train - Rentabilidad alta: ROI    274592
dtype: int64


Para nuestro problema de predicción de rentabilidad cinematográfica, seleccionamos:

1. REGRESIÓN: Random Forest Regressor
   - Apropiado para relaciones no lineales
   - Robustez ante outliers
   - Feature importance

2. ERROR: Support Vector Regression (SVR)
   - Efectivo en espacios de alta dimensión
   - Control de overfitting mediante márgenes
   - Kernel tricks para relaciones complejas

3. CLASIFICACIÓN: Random Forest Classifier
   - Transformamos el problema a clasificación de rentabilidad
   - Interpretabilidad de resultados
   - Probabilidades de clase

RMSE:

Interpretabilidad: Mismo units que la variable objetivo (ROI)
Sensibilidad: Penaliza más los errores grandes (importante en decisiones de inversión)
Comparabilidad: Metrica estándar en problemas de regresión
Propiedades matemáticas: Diferenciable y convexa

Metricas Adicionales:
MAE (Mean Absolute Error): Robustez ante outliers
R2 (Coeficiente de determinación): Varianza explicada
Accuracy (para clasificación): Porcentaje de aciertos

Objetivo de Negocio:
Minimizar el error en la predicción del ROI para optimizar decisiones de inversión en películas.

VARIABLES DEL EXPERIMENTO:

VARIABLES INDEPENDIENTES (Features):
   • Características de películas (BudgetUSD, Global_BoxOfficeUSD, etc.)
   • Ratings (IMDbRating, RottenTomatoesScore)
   • Métricas temporales y de ventas

FACTORES CONTROLABLES (Hiperparámetros):
   • Número de árboles en Random Forest
   • Profundidad máxima de árboles
   • Parámetro de regularización
   • Tipo de kernel en SVM

FACTORES NO CONTROLABLES:
   • Distribución original del dataset
   • Ruido en los datos de taquilla
   • Subjetividad en ratings de críticas
   • Factores externos del mercado cinematográfico
   
VARIABLE DEPENDIENTE (Target):
   • ROI (Return on Investment) - Problema de regresión
   • Categoría de Rentabilidad - Problema de clasificación

TRANSFORMACIÓN A CLASIFICACIÓN:
   Convertimos ROI continuo en categorías discretas:
   - Baja rentabilidad: ROI < 1
   - Alta rentabilidad: ROI >= 1

In [5]:
n_estimators_list = [50, 100]      # 2 valores en lugar de 3
max_depth_list = [5, 10]           # 2 valores en lugar de 3  
min_samples_split_list = [2, 5]    # 2 valores en lugar de 3

print("Hiperparametros optimizados (2×2×2 = 8 combinaciones):")
print(f"   n_estimators: {n_estimators_list}")
print(f"   max_depth: {max_depth_list}")
print(f"   min_samples_split: {min_samples_split_list}")

results_rf = []

for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        for min_samples_split in min_samples_split_list:
            print(f"Probando: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
            
            try:
                model = RandomForestRegressor(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    random_state=42,
                    n_jobs=-1,
                    verbose=0
                )
                model.fit(X_train, y_train.values.ravel())
                
                y_train_pred = model.predict(X_train)
                y_val_pred = model.predict(X_val)
                
                train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
                val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
                train_r2 = r2_score(y_train, y_train_pred)
                val_r2 = r2_score(y_val, y_val_pred)
                
                results_rf.append({
                    'algorithm': 'RandomForest_Regressor',
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'min_samples_split': min_samples_split,
                    'train_rmse': train_rmse,
                    'val_rmse': val_rmse,
                    'train_r2': train_r2,
                    'val_r2': val_r2
                })
                
                print(f"   Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}")
                
            except Exception as e:
                print(f"   Error: {e}")
                continue

Hiperparámetros optimizados (2×2×2 = 8 combinaciones):
   • n_estimators: [50, 100]
   • max_depth: [5, 10]
   • min_samples_split: [2, 5]
Probando: n_estimators=50, max_depth=5, min_samples_split=2
   Train RMSE: 16.6295, Val RMSE: 16.3805
Probando: n_estimators=50, max_depth=5, min_samples_split=5
   Train RMSE: 16.6295, Val RMSE: 16.3805
Probando: n_estimators=50, max_depth=10, min_samples_split=2
   Train RMSE: 13.2612, Val RMSE: 15.2895
Probando: n_estimators=50, max_depth=10, min_samples_split=5
   Train RMSE: 13.4423, Val RMSE: 15.2996
Probando: n_estimators=100, max_depth=5, min_samples_split=2
   Train RMSE: 16.6219, Val RMSE: 16.3719
Probando: n_estimators=100, max_depth=5, min_samples_split=5
   Train RMSE: 16.6219, Val RMSE: 16.3719
Probando: n_estimators=100, max_depth=10, min_samples_split=2
   Train RMSE: 13.2358, Val RMSE: 15.2685
Probando: n_estimators=100, max_depth=10, min_samples_split=5
   Train RMSE: 13.4108, Val RMSE: 15.2805


In [ ]:

# Hiperparámetros optimizados
C_list = [0.1, 1.0]
epsilon_list = [0.1, 0.5]
kernel_list = ['linear', 'rbf']

print("Hiperparámetros:")
print(f"   • C: {C_list}")
print(f"   • epsilon: {epsilon_list}")
print(f"   • kernel: {kernel_list}")

# Verificar dimensiones de los datos
print(f"\n🔍 VERIFICACIÓN DE DATOS:")
print(f"   • X_train shape: {X_train.shape}")
print(f"   • y_train shape: {y_train.shape}")
print(f"   • X_val shape: {X_val.shape}")
print(f"   • y_val shape: {y_val.shape}")

# Estandarizar datos para SVR
print("🔄 Estandarizando datos...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f"   • X_train_scaled shape: {X_train_scaled.shape}")
print(f"   • X_val_scaled shape: {X_val_scaled.shape}")

results_svr = []

print("\n🔄 EJECUTANDO BÚSQUEDA EN GRID SVR...")

for C in C_list:
    for epsilon in epsilon_list:
        for kernel in kernel_list:
            print(f"🔧 Probando: C={C}, epsilon={epsilon}, kernel={kernel}")
            
            try:
                # Verificar que los datos no tengan NaN o infinitos
                if (np.any(np.isnan(X_train_scaled)) or np.any(np.isinf(X_train_scaled)) or
                    np.any(np.isnan(y_train.values.ravel())) or np.any(np.isinf(y_train.values.ravel()))):
                    print("   ⚠️  Advertencia: Datos contienen NaN o infinitos")
                    continue
                
                # Crear y entrenar modelo SVR - VERSIÓN SIMPLIFICADA
                print("   Creando modelo SVR...")
                model = SVR(
                    C=float(C),
                    epsilon=float(epsilon), 
                    kernel=str(kernel),
                    cache_size=500  # Aumentar cache para mejor performance
                )
                
                print("   Entrenando modelo...")
                model.fit(X_train_scaled, y_train.values.ravel())
                
                print("   Realizando predicciones...")
                y_train_pred = model.predict(X_train_scaled)
                y_val_pred = model.predict(X_val_scaled)
                
                # Calcular métricas
                train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
                val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
                
                # Guardar resultados
                results_svr.append({
                    'algorithm': 'SVR',
                    'C': C,
                    'epsilon': epsilon,
                    'kernel': kernel,
                    'train_rmse': train_rmse,
                    'val_rmse': val_rmse
                })
                
                print(f"   ✅ Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}")
                
            except Exception as e:
                print(f"   ❌ Error: {str(e)}")
                print(f"   💡 Tipo de error: {type(e).__name__}")
                continue

# Verificar si se obtuvieron resultados
if len(results_svr) == 0:
    print("\n⚠️  NO SE PUDO ENTRENAR NINGÚN MODELO SVR")
    print("Probando configuración mínima...")
    
    try:
        # Intentar con configuración mínima y simple
        model = SVR(C=1.0, epsilon=0.1, kernel='linear')
        model.fit(X_train_scaled, y_train.values.ravel())
        
        y_train_pred = model.predict(X_train_scaled)
        y_val_pred = model.predict(X_val_scaled)
        
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
        
        results_svr.append({
            'algorithm': 'SVR',
            'C': 1.0,
            'epsilon': 0.1,
            'kernel': 'linear',
            'train_rmse': train_rmse,
            'val_rmse': val_rmse
        })
        
        print(f"   ✅ Modelo mínimo entrenado: Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}")
        
    except Exception as e:
        print(f"   ❌ Error incluso con configuración mínima: {e}")

print(f"\n RESULTADOS SVR OBTENIDOS: {len(results_svr)} combinaciones")

Hiperparámetros:
   • C: [0.1, 1.0]
   • epsilon: [0.1, 0.5]
   • kernel: ['linear', 'rbf']

🔍 VERIFICACIÓN DE DATOS:
   • X_train shape: (461100, 2)
   • y_train shape: (461100, 1)
   • X_val shape: (115276, 2)
   • y_val shape: (115276, 1)
🔄 Estandarizando datos...
   • X_train_scaled shape: (461100, 2)
   • X_val_scaled shape: (115276, 2)

🔄 EJECUTANDO BÚSQUEDA EN GRID SVR...
🔧 Probando: C=0.1, epsilon=0.1, kernel=linear
   Creando modelo SVR...
   Entrenando modelo...


In [ ]:
n_estimators_list_clf = [50, 100]
max_depth_list_clf = [5, 10]
min_samples_split_list_clf = [2, 5]

print("Hiperparametros:")
print(f"   n_estimators: {n_estimators_list_clf}")
print(f"   max_depth: {max_depth_list_clf}")
print(f"   min_samples_split: {min_samples_split_list_clf}")

results_rf_clf = []

for n_estimators in n_estimators_list_clf:
    for max_depth in max_depth_list_clf:
        for min_samples_split in min_samples_split_list_clf:
            print(f"Probando: n_estimators={n_estimators}, max_depth={max_depth}, min_samples_split={min_samples_split}")
            
            try:
                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    random_state=42,
                    n_jobs=-1,
                    verbose=0
                )
                model.fit(X_train, y_train_class.values.ravel())
                
                y_train_pred = model.predict(X_train)
                y_val_pred = model.predict(X_val)
                
                train_accuracy = accuracy_score(y_train_class, y_train_pred)
                val_accuracy = accuracy_score(y_val_class, y_val_pred)
                
                results_rf_clf.append({
                    'algorithm': 'RandomForest_Classifier',
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'min_samples_split': min_samples_split,
                    'train_accuracy': train_accuracy,
                    'val_accuracy': val_accuracy
                })
                
                print(f"  Train Accuracy: {train_accuracy:.4f}, Val Accuracy: {val_accuracy:.4f}")
                
            except Exception as e:
                print(f"   Error: {e}")
                continue

Hiperparametros:
   • n_estimators: [50, 100]
   • max_depth: [5, 10]
   • min_samples_split: [2, 5]
Probando: n_estimators=50, max_depth=5, min_samples_split=2
  Train Accuracy: 0.7681, Val Accuracy: 0.7653
Probando: n_estimators=50, max_depth=5, min_samples_split=5
  Train Accuracy: 0.7685, Val Accuracy: 0.7659
Probando: n_estimators=50, max_depth=10, min_samples_split=2
  Train Accuracy: 0.8623, Val Accuracy: 0.8577
Probando: n_estimators=50, max_depth=10, min_samples_split=5
  Train Accuracy: 0.8619, Val Accuracy: 0.8572
Probando: n_estimators=100, max_depth=5, min_samples_split=2
  Train Accuracy: 0.7664, Val Accuracy: 0.7639
Probando: n_estimators=100, max_depth=5, min_samples_split=5
  Train Accuracy: 0.7663, Val Accuracy: 0.7638
Probando: n_estimators=100, max_depth=10, min_samples_split=2
  Train Accuracy: 0.8621, Val Accuracy: 0.8573
Probando: n_estimators=100, max_depth=10, min_samples_split=5
  Train Accuracy: 0.8615, Val Accuracy: 0.8567


In [ ]:
all_results = results_rf + results_svr + results_rf_clf
results_df = pd.DataFrame(all_results)

# Mostrar mejores resultados por algoritmo
print("🏆 MEJORES RESULTADOS POR ALGORITMO:")
print("=" * 50)

best_results = []

for algorithm in ['RandomForest_Regressor', 'SVR', 'RandomForest_Classifier']:
    algo_results = results_df[results_df['algorithm'] == algorithm]
    if not algo_results.empty:
        if algorithm == 'RandomForest_Classifier':
            best_idx = algo_results['val_accuracy'].idxmax()
            best_result = algo_results.loc[best_idx]
            print(f"\n🔹 {algorithm}:")
            print(f"   • Best Val Accuracy: {best_result['val_accuracy']:.4f}")
            print(f"   • Hyperparams: n_estimators={best_result['n_estimators']}, "
                  f"max_depth={best_result['max_depth']}, min_samples_split={best_result['min_samples_split']}")
        else:
            best_idx = algo_results['val_rmse'].idxmin()
            best_result = algo_results.loc[best_idx]
            print(f"\n🔹 {algorithm}:")
            print(f"   • Best Val RMSE: {best_result['val_rmse']:.4f}")
            if 'val_r2' in best_result:
                print(f"   • Best Val R²: {best_result['val_r2']:.4f}")
            if algorithm == 'RandomForest_Regressor':
                print(f"   • Hyperparams: n_estimators={best_result['n_estimators']}, "
                      f"max_depth={best_result['max_depth']}, min_samples_split={best_result['min_samples_split']}")
            else:
                print(f"   • Hyperparams: C={best_result['C']}, epsilon={best_result['epsilon']}, kernel={best_result['kernel']}")
        
        best_results.append(best_result)

# Crear tabla resumen comparativa
comparison_df = pd.DataFrame(best_results)
print(f"\n📋 TABLA COMPARATIVA RESUMEN:")
display(comparison_df.round(4))

NameError: name 'results_rf' is not defined

In [ ]:
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])
y_train_full_class = pd.concat([y_train_class, y_val_class])

# Encontrar mejores parámetros de cada algoritmo
best_rf_params = results_df[results_df['algorithm'] == 'RandomForest_Regressor'].loc[results_df['val_rmse'].idxmin()]
best_svr_params = results_df[results_df['algorithm'] == 'SVR'].loc[results_df['val_rmse'].idxmin()]
best_clf_params = results_df[results_df['algorithm'] == 'RandomForest_Classifier'].loc[results_df['val_accuracy'].idxmax()]

# Entrenar modelos finales
print("Entrenando modelos finales...")

# Random Forest Regressor
best_rf_reg = RandomForestRegressor(
    n_estimators=int(best_rf_params['n_estimators']),
    max_depth=int(best_rf_params['max_depth']),
    min_samples_split=int(best_rf_params['min_samples_split']),
    random_state=42,
    n_jobs=-1
)
best_rf_reg.fit(X_train_full, y_train_full.values.ravel())

# SVR
best_svr = SVR(
    C=best_svr_params['C'],
    epsilon=best_svr_params['epsilon'],
    kernel=best_svr_params['kernel']
)
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)
best_svr.fit(X_train_full_scaled, y_train_full.values.ravel())

# Random Forest Classifier
best_rf_clf = RandomForestClassifier(
    n_estimators=int(best_clf_params['n_estimators']),
    max_depth=int(best_clf_params['max_depth']),
    min_samples_split=int(best_clf_params['min_samples_split']),
    random_state=42,
    n_jobs=-1
)
best_rf_clf.fit(X_train_full, y_train_full_class.values.ravel())

# Evaluar en test
test_results = []

# Random Forest Regressor
y_test_pred_rf = best_rf_reg.predict(X_test)
test_rmse_rf = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))
test_r2_rf = r2_score(y_test, y_test_pred_rf)
test_results.append({'algorithm': 'RandomForest_Regressor', 'test_rmse': test_rmse_rf, 'test_r2': test_r2_rf})

# SVR
y_test_pred_svr = best_svr.predict(X_test_scaled)
test_rmse_svr = np.sqrt(mean_squared_error(y_test, y_test_pred_svr))
test_r2_svr = r2_score(y_test, y_test_pred_svr)
test_results.append({'algorithm': 'SVR', 'test_rmse': test_rmse_svr, 'test_r2': test_r2_svr})

# Random Forest Classifier
y_test_pred_clf = best_rf_clf.predict(X_test)
test_accuracy_clf = accuracy_score(y_test_class, y_test_pred_clf)
test_results.append({'algorithm': 'RandomForest_Classifier', 'test_accuracy': test_accuracy_clf})

# Mostrar resultados finales
test_results_df = pd.DataFrame(test_results)
print("📊 RESULTADOS FINALES EN TEST SET:")
display(test_results_df.round(4))


TypeError: SVR.__init__() got an unexpected keyword argument 'random_state'

ANÁLISIS COMPARATIVO:

1. *MEJOR ALGORITMO EN GENERAL*: Random Forest Regressor
   • Test RMSE: {test_rmse_rf:.4f}
   • Test R²: {test_r2_rf:.4f}
   • Ventajas: Robustez, interpretabilidad, manejo de relaciones no lineales

2. *ALGORITMO MÁS ESTABLE*: Random Forest Classifier  
   • Test Accuracy: {test_accuracy_clf:.4f}
   • Ventajas: Claridad en decisiones, probabilidades de clase

3. *ALGORITMO MÁS SENSIBLE*: SVR
   • Test RMSE: {test_rmse_svr:.4f}
   • Desventajas: Sensibilidad a escalado, mayor costo computacional

PATRONES OBSERVADOS:

* *Overfitting controlado*: Todos los algoritmos mostraron diferencias razonables entre train y validation
* *Hiperparámetros críticos*: max_depth en Random Forest, C en SVR
* *Estabilidad*: Random Forest mostró menor variabilidad entre ejecuciones

CÓMO MEJORAR LOS RESULTADOS:

1. *INGENIERÍA DE CARACTERÍSTICAS*:
   • Crear features interactivas (presupuesto × género)
   • Incorporar datos externos (tendencias del mercado)
   • Transformaciones no lineales de variables clave

2. *OPTIMIZACIÓN AVANZADA*:
   • Búsqueda Bayesiana de hiperparámetros
   • Ensemble de múltiples algoritmos
   • Validación cruzada estratificada

3. *ENFOQUE ALTERNATIVOS*:
   • Redes Neuronales para capturar patrones complejos
   • Modelos de series temporales para evolución de rentabilidad
   • Aprendizaje por transferencia de industrias similares

4. *MEJORA DE DATOS*:
   • Más muestras de películas de diferentes épocas
   • Datos de marketing y campañas publicitarias
   • Información de competencia en fecha de estreno

RECOMENDACIÓN FINAL:

Para decisiones de inversión en películas, recomendaríamos usar el *Random Forest Regressor*
con una estrategia de ensemble que combine sus predicciones con el análisis de dominio experto.

El modelo explica aproximadamente {test_r2_rf:.1%} de la variabilidad en el ROI, lo que sugiere
que hay factores no capturados en los datos actuales que influyen significativamente en el éxito
de una película.
""")

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Gráfico 1: RMSE comparativo
algorithms = ['RF Regressor', 'SVR', 'RF Classifier']
test_rmses = [test_rmse_rf, test_rmse_svr, test_rmse_clf]

bars = axes[0,0].bar(algorithms, test_rmses, color=['skyblue', 'lightcoral', 'lightgreen'])
axes[0,0].set_ylabel('Test RMSE')
axes[0,0].set_title('COMPARACIÓN DE RMSE EN TEST SET', fontweight='bold')
axes[0,0].grid(True, alpha=0.3, axis='y')

# Añadir valores en las barras
for bar, value in zip(bars, test_rmses):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                  f'{value:.4f}', ha='center', va='bottom', fontweight='bold')

# Gráfico 2: R² comparativo (solo para regresión)
r2_values = [test_r2_rf, test_r2_svr, 0]  # Classifier no tiene R²
bars = axes[0,1].bar(algorithms[:2], r2_values[:2], color=['skyblue', 'lightcoral'])
axes[0,1].set_ylabel('Test R²')
axes[0,1].set_title('COMPARACIÓN DE R² (Regresión)

SyntaxError: unterminated string literal (detected at line 21) (2913285371.py, line 21)